In [2]:
#Task 1: Data Preparation and Missing Value Handling

import pandas as pd

#Students DataFrame:

students_data = {
    'student_id': [101, 102, 103, 104, 105, 106, 107],
    'name': ['Alice', 'Bob', None, 'David', 'Emma', 'Frank', 'Grace'],
    'email': ['alice@email.com', 'bob@email.com', 'charlie@email.com', None, 'emma@email.com', 'frank@email.com', 'grace@email.com'],
    'city': ['Mumbai', 'Delhi', 'Bangalore', 'Mumbai', None, 'Chennai', 'Delhi']
}



#Enrollments DataFrame:

enrollments_data = {
    'student_id': [101, 102, 103, 105, 108, 109],
    'course_name': ['Python', 'Data Science', 'Python', 'Machine Learning', 'AI', 'Python'],
    'enrollment_date': ['2024-01-15', '2024-01-20', '2024-02-01', '2024-02-10', '2024-02-15', '2024-03-01']
}

#Scores DataFrame:

scores_data = {
    'student_id': [101, 102, 104, 105, 106],
    'exam_score': [85, 92, 78, 88, 95]
}

#Create all three DataFrames
students_df = pd.DataFrame(students_data)
enrollments_df = pd.DataFrame(enrollments_data)
scores_df = pd.DataFrame(scores_data)

#Original Students DataFrame:
print("\nOriginal Students DataFrame:")
print(students_df)

#Display null value count
null_count = students_df.isnull().sum()

#Display null percentage
null_percentage = (students_df.isnull().sum() / len(students_df)) * 100

print("\nNull Value Analysis:")
for col in students_df.columns:
    print(f"Column: {col}, Nulls: {null_count[col]} ({null_percentage[col]:.2f}%)")

#Fill missing 'city' values
students_df['city'] = students_df['city'].fillna('Unknown')

#Drop rows where 'name' is missing
students_df = students_df.dropna(subset=['name'])

#Display cleaned DataFrame
print(f"\nCleaned Students DataFrame ({len(students_df)} rows)")
print(students_df)





Original Students DataFrame:
   student_id   name              email       city
0         101  Alice    alice@email.com     Mumbai
1         102    Bob      bob@email.com      Delhi
2         103   None  charlie@email.com  Bangalore
3         104  David               None     Mumbai
4         105   Emma     emma@email.com       None
5         106  Frank    frank@email.com    Chennai
6         107  Grace    grace@email.com      Delhi

Null Value Analysis:
Column: student_id, Nulls: 0 (0.00%)
Column: name, Nulls: 1 (14.29%)
Column: email, Nulls: 1 (14.29%)
Column: city, Nulls: 1 (14.29%)

Cleaned Students DataFrame (6 rows)
   student_id   name            email     city
0         101  Alice  alice@email.com   Mumbai
1         102    Bob    bob@email.com    Delhi
3         104  David             None   Mumbai
4         105   Emma   emma@email.com  Unknown
5         106  Frank  frank@email.com  Chennai
6         107  Grace  grace@email.com    Delhi


In [ ]:
#Task 2: Multiple Join Operations

#Inner Join: Merge students and enrollments on student_id
inner_df = pd.merge(students_df, enrollments_df, on="student_id", how="inner")
print(f"\nInner Join Result ({len(inner_df)} rows)")

#Print only matching students
result = inner_df[['student_id', 'name', 'course_name', 'enrollment_date']]
print(result.to_string())

excluded_students = students_df.loc[
    ~students_df['student_id'].isin(enrollments_df['student_id']),
    'student_id'
].tolist()

print(f"\nExcluded students: {excluded_students} - Not in enrollments table")

#Left Join: Merge students and enrollments on student_id
left_df = pd.merge(students_df, enrollments_df, on="student_id", how="left")
print(f"\nLeft Join Result ({len(left_df)} rows)")

result_left = left_df[['student_id', 'name', 'course_name', 'enrollment_date']]
print(result_left.to_string())

#Print only matching students
null_students = left_df[left_df['course_name'].isnull()]['student_id'].tolist()
print(f"Students with null course_name: {null_students}")

#Right Join: Merge students and enrollments on student_id
right_df = pd.merge(students_df, enrollments_df, on="student_id", how="right")
print(f"\nRight Join Result ({len(right_df)} rows)")

result_right = right_df[['student_id', 'name', 'course_name', 'enrollment_date']]
print(result_right.to_string())

#Print only missing student ID
missing_names = right_df[right_df['name'].isnull()]['student_id'].tolist()
print("Student IDs without names:", missing_names)

#Outer Join: Merge students and enrollments on student_id
outer_df = pd.merge(students_df, enrollments_df, on="student_id", how="outer")
print(f"\nOuter Join Result ({len(outer_df)} rows)")

#Display rows where either student name is null OR course_name is null
null_rows = outer_df[
    outer_df['name'].isnull() | outer_df['course_name'].isnull()
]


print(null_rows)

#Show (_merge column)
outer_df = pd.merge(
    students_df,
    enrollments_df,
    on="student_id",
    how="outer",
    indicator=True
)

print(outer_df)

print(outer_df[outer_df['_merge'] != 'both'])

print(outer_df['_merge'].value_counts())




Inner Join Result (3 rows)
   student_id   name       course_name enrollment_date
0         101  Alice            Python      2024-01-15
1         102    Bob      Data Science      2024-01-20
2         105   Emma  Machine Learning      2024-02-10

Excluded students: [104, 106, 107] - Not in enrollments table

Left Join Result (6 rows)
   student_id   name       course_name enrollment_date
0         101  Alice            Python      2024-01-15
1         102    Bob      Data Science      2024-01-20
2         104  David               NaN             NaN
3         105   Emma  Machine Learning      2024-02-10
4         106  Frank               NaN             NaN
5         107  Grace               NaN             NaN
Students with null course_name: [104, 106, 107]

Right Join Result (6 rows)
   student_id   name       course_name enrollment_date
0         101  Alice            Python      2024-01-15
1         102    Bob      Data Science      2024-01-20
2         103    NaN            Pyth

In [11]:
#Task 3: Lookup Operation and Automation

#Lookup Operation:
#Create a dictionary mapping -> exam_score
score_dict = scores_df.set_index('student_id')['exam_score'].to_dict()

#Use the .map() function to add exam scores to the students DataFrame
students_df['exam_score'] = students_df['student_id'].map(score_dict)

#Display students with their scores (showing NaN for students without scores)
print("\nLookup Operation Result:")
print(students_df)

#Performance Comparison:
#Implement the same score addition using pd.merge() with a left join

result = pd.merge(students_df,
    scores_df,
    on="student_id",
    how="left"
)

print(result)

#Explain why lookup (map) is more efficient than merge for this scenario
#works like a dictionary lookup map()

score_map = scores_df.set_index("student_id")["exam_score"]

students_df["score"] = students_df["student_id"].map(score_map)





Lookup Operation Result:
   student_id   name            email     city  exam_score
0         101  Alice  alice@email.com   Mumbai        85.0
1         102    Bob    bob@email.com    Delhi        92.0
3         104  David             None   Mumbai        78.0
4         105   Emma   emma@email.com  Unknown        88.0
5         106  Frank  frank@email.com  Chennai        95.0
6         107  Grace  grace@email.com    Delhi         NaN
   student_id   name            email     city  exam_score_x  exam_score_y
0         101  Alice  alice@email.com   Mumbai          85.0          85.0
1         102    Bob    bob@email.com    Delhi          92.0          92.0
2         104  David             None   Mumbai          78.0          78.0
3         105   Emma   emma@email.com  Unknown          88.0          88.0
4         106  Frank  frank@email.com  Chennai          95.0          95.0
5         107  Grace  grace@email.com    Delhi           NaN           NaN


In [19]:
#Create a function auto_merge



def auto_merge(df1, df2, join_type, key_column):
    merged_df = pd.merge(df1, df2, how=join_type, on=key_column)

    result = {
        "result_df": merged_df,
        "row_count": len(merged_df),
        "join_type": join_type
    }

    return result

join_type = input("Enter join type (left/right/inner/outer): ")
key_column = input("Enter key column: ")

result = auto_merge(students_df, scores_df, join_type, key_column)

print("Join Type:", result["join_type"])
print("Row Count:", result["row_count"])
print("Merged DataFrame:")
print(result["result_df"])

Enter join type (left/right/inner/outer): inner
Enter key column: student_id
Join Type: inner
Row Count: 5
Merged DataFrame:
   student_id   name            email     city  exam_score_x  score  \
0         101  Alice  alice@email.com   Mumbai          85.0   85.0   
1         102    Bob    bob@email.com    Delhi          92.0   92.0   
2         104  David             None   Mumbai          78.0   78.0   
3         105   Emma   emma@email.com  Unknown          88.0   88.0   
4         106  Frank  frank@email.com  Chennai          95.0   95.0   

   exam_score_y  
0            85  
1            92  
2            78  
3            88  
4            95  
